# A4 — Estudo de Caso: COVID-19 em Radiografias de Tórax

**Projeto:** EcoVision AI  
**Atividade:** 4.1 — Detecção de COVID-19 em Radiografias de Tórax  
**Abordagens:** ResNet-18 + Conditional GAN + Synthetic Augmentation

## Objetivos

Este notebook reproduz e documenta o estudo de caso de ponta a ponta:

1. reconstruir um dataset com a distribuição descrita no enunciado;
2. diagnosticar problemas metodológicos do projeto original;
3. construir um baseline mais controlado;
4. medir o desempenho por classe, com foco em **recall COVID-19**;
5. implementar uma **Conditional GAN (cGAN)**;
6. diagnosticar **mode collapse / instabilidade adversarial**;
7. testar estratégias de mitigação;
8. adicionar **Mode-Seeking Regularization**;
9. gerar imagens COVID sintéticas;
10. retreinar o mesmo classificador com os dados sintéticos;
11. comparar o recall COVID com e sem augmentation sintética;
12. propor um plano integrado de melhoria.

## Requisitos de execução

- **Ambiente recomendado:** Google Colab com GPU T4
- **GPU:** T4 (16 GB de VRAM)
- **RAM recomendada:** 8 GB ou mais
- **Espaço livre recomendado:** 4 GB ou mais
- **Dataset final:** 1.200 radiografias
- **Classes:** Normal, Pneumonia, COVID
- **Classificador:** ResNet-18 pré-treinada
- **GAN:** Conditional DCGAN em 64 × 64 grayscale
- **Batch size:** 32
- **Seed:** 42
- **Tempo estimado com checkpoints existentes:** ~5–10 minutos
- **Tempo estimado em execução completa no Colab T4:** aproximadamente 20–45 minutos, dependendo de downloads/cache

> Este experimento é acadêmico. Os resultados apresentados não representam validação clínica nem autorização para uso diagnóstico.

## 1. Cenário do estudo de caso

O projeto original descrito na atividade possuía:

```text
Normal:      840
Pneumonia:   240
COVID-19:    120
Total:     1.200
```

Metodologia relatada:

```text
split 80/20 sem estratificação
ResNet-18 sem pré-treinamento
SGD
learning rate fixo = 0.01
15 epochs
sem data augmentation
avaliação apenas por accuracy global
```

Resultado reportado:

```text
Train Accuracy: 93%
Validation Accuracy: 61%
```

Esse cenário é usado como ponto de partida para a análise crítica e para nossos experimentos.

## 2. Preparação do ambiente

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kagglehub",
        "scikit-learn",
        "matplotlib",
        "pandas",
        "pillow",
    ],
    check=True,
)

print("Dependências verificadas.")

In [ ]:
from pathlib import Path
import os
import sys

def encontrar_raiz_projeto():
    candidatos = []

    cwd = Path.cwd().resolve()
    candidatos.extend([cwd, *cwd.parents])

    candidatos.extend([
        Path("/content/EcoVision-AI-organized"),
        Path("/content/EcoVision-AI"),
    ])

    for candidato in candidatos:
        if (
            (candidato / "src" / "xray").exists()
            and (candidato / "src" / "common").exists()
        ):
            return candidato

    try:
        from google.colab import drive

        drive.mount("/content/drive")

        mydrive = Path("/content/drive/MyDrive")

        for nome in ["EcoVision-AI-organized", "EcoVision-AI"]:
            for candidato in mydrive.glob(f"**/{nome}"):
                if (
                    (candidato / "src" / "xray").exists()
                    and (candidato / "src" / "common").exists()
                ):
                    return candidato.resolve()

    except Exception:
        pass

    raise FileNotFoundError(
        "Não encontrei a raiz do EcoVision-AI. "
        "No Colab, coloque a pasta do projeto em /content "
        "ou no Google Drive."
    )

PROJECT_ROOT = encontrar_raiz_projeto()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Diretório atual:", Path.cwd())

In [ ]:
import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {total_vram:.2f} GB")
else:
    print(
        "O notebook funciona em CPU, mas os treinamentos "
        "da ResNet e das GANs ficam consideravelmente mais lentos."
    )

## 3. Preparação do dataset

Para reproduzir a distribuição do estudo de caso, utilizamos o **COVID-19 Radiography Database** e selecionamos:

```text
Normal            840
Viral Pneumonia   240 → renomeada para Pneumonia
COVID             120
```

Antes da amostragem:

1. as imagens são validadas com Pillow;
2. é calculado SHA-256;
3. duplicatas internas são removidas;
4. duplicatas exatas entre classes são impedidas;
5. uma amostra determinística é selecionada com `seed = 42`.

No experimento original do EcoVision, o conjunto final terminou com:

```text
1.200 imagens
1.200 hashes SHA-256 únicos
```

In [ ]:
XRAY_DATASET = PROJECT_ROOT / "data" / "xray_covid"

def dataset_xray_pronto():
    if not XRAY_DATASET.exists():
        return False

    contagens = {}

    for classe in ["COVID", "Normal", "Pneumonia"]:
        pasta = XRAY_DATASET / classe

        if not pasta.exists():
            return False

        contagens[classe] = sum(
            1
            for arquivo in pasta.iterdir()
            if arquivo.is_file()
        )

    return contagens == {
        "COVID": 120,
        "Normal": 840,
        "Pneumonia": 240,
    }

if dataset_xray_pronto():
    print("Dataset de raio-X já está preparado.")
else:
    print("Preparando dataset de raio-X...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.prepare_dataset"],
        cwd=PROJECT_ROOT,
        check=True,
    )

assert dataset_xray_pronto()

print("Dataset validado:", XRAY_DATASET)

## 4. Split estratificado e data augmentation

Em vez do split 80/20 sem estratificação do projeto original, utilizamos:

```text
70% treino
15% validação
15% teste
```

com estratificação.

Isso preserva a proporção de cada classe.

### Transformações de treino

Como se trata de radiografia, as augmentations são deliberadamente leves:

- `RandomRotation(7°)`
- `RandomAffine` com pequena translação e escala
- `ColorJitter` leve de brilho e contraste
- resize 224 × 224
- normalização ImageNet

Validação e teste recebem apenas resize + normalização.

In [ ]:
from src.xray.data_pipeline import (
    localizar_dataset,
    criar_datasets,
    criar_dataloaders,
    contar_classes,
    criar_transforms,
)

dataset_path = localizar_dataset()

(
    dataset_base,
    train_dataset,
    val_dataset,
    test_dataset,
) = criar_datasets(
    dataset_path
)

(
    train_loader,
    val_loader,
    test_loader,
) = criar_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
)

print("Classes:", dataset_base.class_to_idx)
print()
print("Total:", len(dataset_base))
print("Treino:", len(train_dataset), contar_classes(train_dataset))
print("Validação:", len(val_dataset), contar_classes(val_dataset))
print("Teste:", len(test_dataset), contar_classes(test_dataset))

Distribuição reproduzida no experimento:

| Classe | Treino | Validação | Teste |
|---|---:|---:|---:|
| COVID | 84 | 18 | 18 |
| Normal | 588 | 126 | 126 |
| Pneumonia | 168 | 36 | 36 |
| **Total** | **840** | **180** | **180** |

Essa separação será mantida em todos os experimentos. A GAN utiliza **somente os 840 índices de treino**. Validação e teste nunca entram no treinamento generativo.

In [ ]:
train_transform, eval_transform = criar_transforms()

print("Transform de treino:")
print(train_transform)

print()
print("Transform de validação/teste:")
print(eval_transform)

# Parte A — Baseline de classificação

## 5. ResNet-18 pré-treinada

Para criar uma referência mais sólida que o projeto original, utilizamos uma **ResNet-18 pré-treinada no ImageNet**.

A camada final original é substituída:

```text
ResNet-18
   ↓
512 features
   ↓
Linear(512, 3)
   ↓
COVID / Normal / Pneumonia
```

Nesta etapa todo o modelo é fine-tuned.

Configuração:

```text
Loss: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 1e-4
Weight decay: 1e-4
Máximo: 10 epochs
Early stopping patience: 3
```

A loss não recebe pesos de classe propositalmente, porque mais adiante queremos isolar o efeito da **augmentation sintética**.

In [ ]:
import inspect

from src.xray.models.classifier import XRayClassifier

print(inspect.getsource(XRayClassifier))

In [ ]:
modelo_xray = XRayClassifier(
    num_classes=3,
    pretrained=True,
)

entrada = torch.randn(
    2,
    3,
    224,
    224,
)

with torch.no_grad():
    logits = modelo_xray(entrada)

print("Entrada:", entrada.shape)
print("Logits:", logits.shape)

total_params = sum(
    p.numel()
    for p in modelo_xray.parameters()
)

print(f"Parâmetros: {total_params:,}")

assert logits.shape == (2, 3)

print("ResNet-18 adaptada: OK")

In [ ]:
BASELINE_CKPT = (
    PROJECT_ROOT
    / "models"
    / "xray_baseline_best.pth"
)

if BASELINE_CKPT.exists():
    print("Checkpoint baseline já existe. Reutilizando:")
    print(BASELINE_CKPT)
else:
    print("Checkpoint baseline ausente. Iniciando treinamento...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.training.baseline"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
# Avaliação sempre é executada para validar o checkpoint
# no conjunto REAL de teste.

subprocess.run(
    [sys.executable, "-m", "src.xray.evaluation.baseline"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
import json
import pandas as pd

XRAY_RESULTS = PROJECT_ROOT / "results" / "xray"

with open(
    XRAY_RESULTS / "xray_baseline_test_results.json",
    "r",
    encoding="utf-8",
) as arquivo:
    baseline_results = json.load(arquivo)

baseline_table = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "F1 weighted",
        "F1 macro",
        "Macro recall",
        "COVID recall",
    ],
    "Valor (%)": [
        baseline_results["accuracy"] * 100,
        baseline_results["f1_weighted"] * 100,
        baseline_results["f1_macro"] * 100,
        baseline_results["recall_macro"] * 100,
        baseline_results["covid_recall"] * 100,
    ],
})

baseline_table

### Resultado do baseline

O classificador atingiu:

```text
Accuracy global: 95,00%
COVID Recall:     55,56%
COVID detectados: 10/18
COVID perdidos:    8/18
```

Esse resultado mostra por que **accuracy global isolada é perigosa em um dataset desbalanceado**.

O modelo classifica corretamente quase todas as imagens Normal e Pneumonia, elevando a accuracy global. Entretanto, deixa de reconhecer **8 dos 18 casos COVID** no teste.

### Precision × Recall

Para COVID:

```text
Precision = 100%
Recall    = 55,56%
```

Interpretação:

- quando o modelo prevê COVID, ele acerta os casos previstos;
- porém ele **não encontra todos os COVID existentes**.

Em um cenário de triagem, os falsos negativos merecem atenção especial porque representam casos positivos que o sistema deixa passar.

In [ ]:
from IPython.display import display, Image as IPyImage

for nome in [
    "xray_baseline_loss.png",
    "xray_baseline_accuracy.png",
    "xray_baseline_covid_recall.png",
    "xray_baseline_confusion_matrix.png",
]:
    caminho = XRAY_RESULTS / nome

    if caminho.exists():
        print(nome)
        display(IPyImage(filename=str(caminho)))

# Parte B — Conditional GAN

## 6. Por que usar uma GAN?

O conjunto de treino possui:

```text
COVID       84
Pneumonia  168
Normal     588
```

COVID é a classe com menos exemplos.

A hipótese experimental é:

> Se conseguirmos gerar exemplos COVID úteis e variados, talvez o classificador aprenda melhor a classe minoritária.

Isso precisa ser testado, e não assumido. Dados sintéticos só ajudam se acrescentarem informação útil.

## 7. Como funciona uma GAN?

Uma GAN possui dois modelos.

### Generator (G)

Recebe ruído aleatório:

```text
z ~ N(0,1)
```

e tenta gerar uma imagem falsa.

### Discriminator (D)

Recebe uma imagem e tenta distinguir:

```text
real
vs
falsa
```

O treinamento é adversarial:

```text
Generator
   ↓ cria fake
Discriminator
   ↓ tenta detectar
Generator recebe gradiente
   ↓
aprende a enganar melhor o Discriminator
```

### Por que Conditional GAN?

Na cGAN também fornecemos a classe:

```text
z + label COVID
        ↓
Generator
        ↓
radiografia COVID sintética
```

O Discriminator também recebe a classe. Assim ele não pergunta apenas:

```text
"essa radiografia parece real?"
```

mas:

```text
"essa radiografia parece real PARA ESTA CLASSE?"
```

## 8. Arquitetura da cGAN

### Generator

Entrada:

```text
ruído z:          100 dimensões
embedding classe: 100 dimensões
```

Depois:

```text
concatenação = 200
       ↓
ConvTranspose
1×1 → 4×4 → 8×8 → 16×16 → 32×32 → 64×64
       ↓
Tanh
       ↓
imagem grayscale em [-1, 1]
```

### Discriminator

A classe é transformada em um mapa `64 × 64` e concatenada à radiografia.

```text
imagem:     1 canal
label map:  1 canal
--------------------
entrada:    2 canais
```

O modelo termina em um **logit**, e por isso utilizamos `BCEWithLogitsLoss`, que combina de forma numericamente estável a função sigmoid com binary cross-entropy.

In [ ]:
from src.xray.models.cgan import (
    ConditionalGenerator,
    ConditionalDiscriminator,
    LATENT_DIM,
)

generator_test = ConditionalGenerator().to(DEVICE)
discriminator_test = ConditionalDiscriminator().to(DEVICE)

noise = torch.randn(
    8,
    LATENT_DIM,
    device=DEVICE,
)

labels = torch.tensor(
    [0, 0, 1, 1, 2, 2, 0, 1],
    device=DEVICE,
)

with torch.no_grad():
    fake = generator_test(
        noise,
        labels,
    )

    logits_d = discriminator_test(
        fake,
        labels,
    )

print("Fake images:", fake.shape)
print("D logits:", logits_d.shape)

params_g = sum(
    p.numel()
    for p in generator_test.parameters()
)

params_d = sum(
    p.numel()
    for p in discriminator_test.parameters()
)

print(f"Generator parameters: {params_g:,}")
print(f"Discriminator parameters: {params_d:,}")

assert fake.shape == (8, 1, 64, 64)
assert logits_d.shape == (8,)

print("Conditional GAN: OK")

## 9. Training loop adversarial correto

Para cada batch, o treinamento ocorre em duas etapas.

### 9.1 Treinar o Discriminator

Primeiro com imagens reais:

```text
D(real, classe)
→ target = 1
```

Depois com imagens falsas:

```text
fake = G(z, classe)

D(fake.detach(), classe)
→ target = 0
```

O `.detach()` impede que, durante esta etapa, os gradientes do Discriminator atualizem o Generator.

A loss do Discriminator é a média:

\[
L_D = rac{L_{real} + L_{fake}}{2}
\]

### 9.2 Treinar o Generator

Agora queremos que as imagens falsas sejam classificadas como reais:

```text
D(G(z, classe), classe)
→ target = 1
```

A loss do Generator é retropropagada até G:

```text
loss_G.backward()
optimizer_G.step()
```

Ou seja, o Generator melhora usando a resposta do Discriminator como sinal de treinamento.

## 10. Dataset específico da GAN

A GAN utiliza:

- exatamente os **840 índices do treino real**;
- nenhuma imagem de validação;
- nenhuma imagem de teste;
- grayscale;
- resolução 64 × 64;
- normalização para `[-1, 1]`.

No treinamento das cGANs utilizamos `WeightedRandomSampler` com peso inverso à frequência de classe, para que COVID não seja quase ignorado pela exposição muito maior à classe Normal.

In [ ]:
from src.xray.gan_data import criar_dataset_gan

dataset_gan_base, dataset_gan = criar_dataset_gan()

print("Imagens usadas pela GAN:", len(dataset_gan))
print("Classes:", dataset_gan_base.class_to_idx)

assert len(dataset_gan) == 840

imagem_gan, label_gan = dataset_gan[0]

print("Shape:", imagem_gan.shape)
print("Min:", float(imagem_gan.min()))
print("Max:", float(imagem_gan.max()))

# Parte C — Experimento 1: cGAN base

## 11. Configuração base

```text
Epochs: 20
Batch: 32
G learning rate: 0.0002
D learning rate: 0.0002
Adam betas: (0.5, 0.999)
Loss: BCEWithLogitsLoss
```

Além das losses, medimos:

```text
D(real)
D(fake)
diversidade COVID
```

### Métrica auxiliar de diversidade

Geramos várias imagens COVID com vetores `z` diferentes e calculamos o desvio padrão médio entre as imagens.

Quanto menor esse valor, maior o sinal de que diferentes ruídos estão produzindo saídas muito parecidas.

Essa métrica **não mede qualidade clínica**. É apenas um indicador auxiliar de diversidade/mode collapse.

In [ ]:
BASE_CGAN_CKPT = (
    PROJECT_ROOT
    / "models"
    / "xray_cgan_base_last.pth"
)

if BASE_CGAN_CKPT.exists():
    print("Checkpoint cGAN base já existe. Reutilizando.")
else:
    print("Treinando cGAN base...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.training.cgan_base"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
with open(
    XRAY_RESULTS / "cgan" / "cgan_base_history.json",
    "r",
    encoding="utf-8",
) as arquivo:
    cgan_base_history = json.load(arquivo)

pd.DataFrame({
    "Métrica": [
        "G loss inicial",
        "G loss final",
        "D loss inicial",
        "D loss final",
        "D(real) final",
        "D(fake) final",
        "Diversidade COVID inicial",
        "Diversidade COVID final",
    ],
    "Valor": [
        cgan_base_history["generator_loss"][0],
        cgan_base_history["generator_loss"][-1],
        cgan_base_history["discriminator_loss"][0],
        cgan_base_history["discriminator_loss"][-1],
        cgan_base_history["discriminator_real_score"][-1],
        cgan_base_history["discriminator_fake_score"][-1],
        cgan_base_history["covid_diversity"][0],
        cgan_base_history["covid_diversity"][-1],
    ],
})

In [ ]:
for epoch in [1, 5, 10, 15, 20]:
    caminho = (
        XRAY_RESULTS
        / "cgan"
        / "samples"
        / f"epoch_{epoch:03d}.png"
    )

    if caminho.exists():
        print(f"cGAN base — epoch {epoch}")
        display(IPyImage(filename=str(caminho)))

## 12. Diagnóstico da cGAN base — mode collapse

No final da cGAN base:

```text
D(real) ≈ 0,7857
D(fake) ≈ 0,2234
D loss  ≈ 0,3047

Diversidade COVID:
0,0427 → 0,0207
```

O Discriminator tornou-se muito mais confiante em distinguir real de falso.

Visualmente, nas épocas posteriores, diferentes vetores de ruído dentro da mesma classe passaram a gerar imagens muito semelhantes.

Esse padrão é compatível com **mode collapse**:

```text
z1 ─┐
z2 ─┤
z3 ─┼→ Generator → praticamente o mesmo protótipo
z4 ─┘
```

Em vez de representar a variedade da distribuição real, o Generator encontra poucos padrões que conseguem competir com o Discriminator.

# Parte D — Experimento 2: primeira mitigação

## 13. Estratégias aplicadas

A segunda cGAN mantém a arquitetura, mas altera o treinamento.

### 13.1 Learning rates assimétricos

```text
Generator LR     = 0.0002
Discriminator LR = 0.0001
```

O Discriminator aprende mais lentamente para reduzir sua dominância.

### 13.2 One-sided label smoothing

Em vez de:

```text
real = 1.0
```

usamos:

```text
real = 0.9
```

Isso reduz a tendência de o Discriminator ficar excessivamente confiante nos exemplos reais.

### 13.3 Instance noise

Adicionamos pequeno ruído gaussiano às imagens vistas por D:

```text
std inicial = 0.05
       ↓
decaimento linear
       ↓
std final = 0
```

A ideia é suavizar inicialmente a tarefa do Discriminator e melhorar o equilíbrio adversarial.

In [ ]:
MITIGATED_CGAN_CKPT = (
    PROJECT_ROOT
    / "models"
    / "xray_cgan_mitigated_last.pth"
)

if MITIGATED_CGAN_CKPT.exists():
    print("Checkpoint cGAN mitigada já existe. Reutilizando.")
else:
    print("Treinando cGAN mitigada...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.training.cgan_mitigated"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
with open(
    XRAY_RESULTS / "cgan_mitigated" / "cgan_mitigated_history.json",
    "r",
    encoding="utf-8",
) as arquivo:
    cgan_mitigated_history = json.load(arquivo)

pd.DataFrame({
    "Métrica": [
        "G loss final",
        "D loss final",
        "D(real) final",
        "D(fake) final",
        "Diversidade COVID inicial",
        "Diversidade COVID final",
    ],
    "Valor": [
        cgan_mitigated_history["generator_loss"][-1],
        cgan_mitigated_history["discriminator_loss"][-1],
        cgan_mitigated_history["discriminator_real_score"][-1],
        cgan_mitigated_history["discriminator_fake_score"][-1],
        cgan_mitigated_history["covid_diversity"][0],
        cgan_mitigated_history["covid_diversity"][-1],
    ],
})

In [ ]:
for epoch in [1, 5, 10, 15, 20]:
    caminho = (
        XRAY_RESULTS
        / "cgan_mitigated"
        / "samples"
        / f"epoch_{epoch:03d}.png"
    )

    if caminho.exists():
        print(f"cGAN mitigada — epoch {epoch}")
        display(IPyImage(filename=str(caminho)))

## 14. Resultado da primeira mitigação

Comparando o final:

```text
cGAN base
D(real) = 0,7857
D(fake) = 0,2234
D loss  = 0,3047

cGAN mitigada
D(real) = 0,6076
D(fake) = 0,2998
D loss  = 0,5475
```

Isso indica um adversarial game menos dominado por D.

Entretanto:

```text
Diversidade final base:      0,0207
Diversidade final mitigada:  0,0189
```

Portanto, **o equilíbrio G/D melhorou, mas a diversidade não**.

Visualmente, as imagens de uma mesma classe continuaram muito parecidas.

Essa etapa é importante metodologicamente: uma técnica pode estabilizar uma parte do treinamento sem resolver o mode collapse.

# Parte E — Experimento 3: Mode-Seeking Regularization

## 15. Atacando diretamente a baixa diversidade

A terceira versão mantém:

- LR menor no Discriminator;
- label smoothing;
- instance noise;

e acrescenta **Mode-Seeking Regularization**.

Selecionamos dois vetores de ruído diferentes:

```text
z1
z2
```

com a mesma classe:

```text
G(z1, classe)
G(z2, classe)
```

Calculamos aproximadamente:

\[
score_{MS} =
rac{distância(G(z_1),G(z_2))}
{distância(z_1,z_2)+\epsilon}
\]

Queremos **maximizar** esse score.

Como os otimizadores minimizam losses:

\[
L_{MS} = -score_{MS}
\]

e:

\[
L_G =
L_{adv}
+
\lambda L_{MS}
\]

com:

```text
lambda = 1.0
```

Assim, o Generator é explicitamente incentivado a responder a diferentes vetores latentes com diferentes imagens.

In [ ]:
DIVERSITY_CGAN_CKPT = (
    PROJECT_ROOT
    / "models"
    / "xray_cgan_diversity_last.pth"
)

if DIVERSITY_CGAN_CKPT.exists():
    print("Checkpoint cGAN Mode-Seeking já existe. Reutilizando.")
else:
    print("Treinando cGAN com Mode-Seeking...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.training.cgan_diversity"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
with open(
    XRAY_RESULTS / "cgan_diversity" / "cgan_diversity_history.json",
    "r",
    encoding="utf-8",
) as arquivo:
    cgan_diversity_history = json.load(arquivo)

pd.DataFrame({
    "Métrica": [
        "G total loss final",
        "G adversarial loss final",
        "D loss final",
        "D(real) final",
        "D(fake) final",
        "Diversidade COVID inicial",
        "Diversidade COVID final",
        "Mode-Seeking inicial",
        "Mode-Seeking final",
    ],
    "Valor": [
        cgan_diversity_history["generator_loss"][-1],
        cgan_diversity_history["generator_adversarial_loss"][-1],
        cgan_diversity_history["discriminator_loss"][-1],
        cgan_diversity_history["discriminator_real_score"][-1],
        cgan_diversity_history["discriminator_fake_score"][-1],
        cgan_diversity_history["covid_diversity"][0],
        cgan_diversity_history["covid_diversity"][-1],
        cgan_diversity_history["mode_seeking_score"][0],
        cgan_diversity_history["mode_seeking_score"][-1],
    ],
})

In [ ]:
for epoch in [1, 5, 10, 15, 20]:
    caminho = (
        XRAY_RESULTS
        / "cgan_diversity"
        / "samples"
        / f"epoch_{epoch:03d}.png"
    )

    if caminho.exists():
        print(f"cGAN Mode-Seeking — epoch {epoch}")
        display(IPyImage(filename=str(caminho)))

In [ ]:
gan_comparison = pd.DataFrame({
    "Experimento": [
        "cGAN base",
        "Primeira mitigação",
        "Mode-Seeking",
    ],
    "Diversidade COVID inicial": [
        cgan_base_history["covid_diversity"][0],
        cgan_mitigated_history["covid_diversity"][0],
        cgan_diversity_history["covid_diversity"][0],
    ],
    "Diversidade COVID final": [
        cgan_base_history["covid_diversity"][-1],
        cgan_mitigated_history["covid_diversity"][-1],
        cgan_diversity_history["covid_diversity"][-1],
    ],
    "D(real) final": [
        cgan_base_history["discriminator_real_score"][-1],
        cgan_mitigated_history["discriminator_real_score"][-1],
        cgan_diversity_history["discriminator_real_score"][-1],
    ],
    "D(fake) final": [
        cgan_base_history["discriminator_fake_score"][-1],
        cgan_mitigated_history["discriminator_fake_score"][-1],
        cgan_diversity_history["discriminator_fake_score"][-1],
    ],
})

gan_comparison

## 16. Interpretação final das três GANs

Diversidade COVID final:

```text
cGAN base              0,0207
Primeira mitigação     0,0189
Mode-Seeking           0,0261
```

A versão Mode-Seeking terminou com diversidade aproximadamente:

- **26% maior que a cGAN base**;
- **38% maior que a primeira mitigação**.

Entretanto, a inspeção visual ainda mostra forte semelhança entre imagens da mesma classe.

Portanto, a conclusão correta é:

> A Mode-Seeking Regularization produziu **melhoria parcial de diversidade**, mas não eliminou o mode collapse.

Outro ponto importante: a maior diversidade da versão Mode-Seeking ocorreu na época 1 (`≈ 0,0678`), quando as imagens ainda eram pouco estruturadas. Logo:

```text
diversidade alta
≠
qualidade alta
```

Qualidade e diversidade precisam ser avaliadas em conjunto.

# Parte F — Synthetic Augmentation

## 17. Quantas imagens sintéticas gerar?

No treino real:

```text
COVID       84
Pneumonia  168
Normal     588
```

Optamos por gerar **84 imagens COVID**, resultando em:

```text
COVID real        84
COVID sintético   84
--------------------
COVID treino     168
```

Assim igualamos COVID à quantidade real de Pneumonia sem inundar o treinamento com centenas de exemplos sintéticos de qualidade limitada.

A geração usa o checkpoint da cGAN com Mode-Seeking.

In [ ]:
SYNTHETIC_COVID_DIR = (
    PROJECT_ROOT
    / "data"
    / "xray_synthetic"
    / "COVID"
)

synthetic_count = (
    sum(
        1
        for arquivo in SYNTHETIC_COVID_DIR.glob("*.png")
    )
    if SYNTHETIC_COVID_DIR.exists()
    else 0
)

if synthetic_count == 84:
    print("84 imagens COVID sintéticas já existem.")
else:
    print("Gerando 84 imagens COVID sintéticas...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.generation.generate_synthetic_covid"],
        cwd=PROJECT_ROOT,
        check=True,
    )

synthetic_count = sum(
    1
    for arquivo in SYNTHETIC_COVID_DIR.glob("*.png")
)

print("COVID sintéticas:", synthetic_count)

assert synthetic_count == 84

In [ ]:
preview = (
    XRAY_RESULTS
    / "synthetic_covid_preview.png"
)

if preview.exists():
    display(
        IPyImage(
            filename=str(preview)
        )
    )

## 18. Experimento controlado com imagens sintéticas

Para testar o impacto da GAN, mantemos o máximo possível constante.

### Baseline

```text
84 COVID reais
588 Normal reais
168 Pneumonia reais
```

### Synthetic augmentation

```text
84 COVID reais
84 COVID sintéticos
588 Normal reais
168 Pneumonia reais
```

Total de treino:

```text
840 → 924 imagens
```

Mantemos:

- ResNet-18;
- pesos ImageNet;
- CrossEntropy sem class weights;
- AdamW;
- learning rate `1e-4`;
- weight decay `1e-4`;
- early stopping;
- mesma validação real;
- mesmo teste real.

Assim, a principal variável experimental é a inclusão dos 84 exemplos sintéticos.

In [ ]:
SYNTHETIC_CKPT = (
    PROJECT_ROOT
    / "models"
    / "xray_synthetic_best.pth"
)

if SYNTHETIC_CKPT.exists():
    print("Checkpoint real + sintético já existe. Reutilizando.")
else:
    print("Treinando classificador com augmentation sintética...")

    subprocess.run(
        [sys.executable, "-m", "src.xray.training.synthetic"],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.xray.evaluation.synthetic"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
with open(
    XRAY_RESULTS / "xray_baseline_vs_synthetic.json",
    "r",
    encoding="utf-8",
) as arquivo:
    comparison = json.load(arquivo)

comparison_table = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Macro F1",
        "Macro Recall",
        "COVID Recall",
    ],
    "Baseline (%)": [
        comparison["baseline"]["accuracy"] * 100,
        comparison["baseline"]["macro_f1"] * 100,
        comparison["baseline"]["macro_recall"] * 100,
        comparison["baseline"]["covid_recall"] * 100,
    ],
    "Com sintético (%)": [
        comparison["synthetic"]["accuracy"] * 100,
        comparison["synthetic"]["macro_f1"] * 100,
        comparison["synthetic"]["macro_recall"] * 100,
        comparison["synthetic"]["covid_recall"] * 100,
    ],
    "Diferença (pp)": [
        comparison["difference"]["accuracy"] * 100,
        comparison["difference"]["macro_f1"] * 100,
        comparison["difference"]["macro_recall"] * 100,
        comparison["difference"]["covid_recall"] * 100,
    ],
})

comparison_table

In [ ]:
for nome in [
    "xray_synthetic_loss.png",
    "xray_synthetic_accuracy.png",
    "xray_synthetic_covid_recall.png",
    "xray_synthetic_confusion_matrix.png",
]:
    caminho = XRAY_RESULTS / nome

    if caminho.exists():
        print(nome)
        display(IPyImage(filename=str(caminho)))

## 19. Resultado central: impacto da augmentation sintética

| Métrica | Baseline | Real + sintético | Diferença |
|---|---:|---:|---:|
| Accuracy | 95,00% | 93,33% | -1,67 pp |
| Macro F1 | 88,86% | 83,35% | -5,51 pp |
| Macro Recall | 84,26% | 78,70% | -5,56 pp |
| **COVID Recall** | **55,56%** | **38,89%** | **-16,67 pp** |

Casos COVID:

```text
Baseline:
10 / 18 detectados
8 / 18 perdidos

Com sintéticos:
7 / 18 detectados
11 / 18 perdidos
```

### Conclusão experimental

Neste experimento, a augmentation sintética **não melhorou** a classe minoritária.

O recall COVID caiu:

```text
55,56%
   ↓
38,89%
```

Essa queda é coerente com a limitação observada na GAN: os exemplos sintéticos possuíam pouca diversidade e sinais de mode collapse.

Portanto:

```text
mais imagens
≠
automaticamente mais informação útil
```

Imagens sintéticas redundantes ou artificiais podem não representar a variabilidade da distribuição real e podem prejudicar a generalização.

## 20. Limitação estatística da comparação

O conjunto de teste contém apenas:

```text
18 imagens COVID
```

Cada caso corresponde a:

\[
1 / 18 pprox 5,56
\]

pontos percentuais de recall.

A diferença de `-16,67 pp` representa exatamente **3 casos a menos detectados**.

Isso é uma evidência válida para o experimento da atividade, mas não permite concluir universalmente que:

```text
"GANs pioram classificação médica"
```

A conclusão correta é restrita a este experimento:

> Com este dataset, esta cGAN, esta qualidade/diversidade sintética e esta configuração de treinamento, a augmentation sintética piorou o recall COVID no teste utilizado.

# Parte G — Diagnóstico do projeto original

## 21. Problemas técnicos e impactos esperados

A atividade exige pelo menos cinco. Aqui identificamos sete.

| Problema | Diagnóstico técnico | Impacto esperado |
|---|---|---|
| **Dataset desbalanceado** | 840 Normal, 240 Pneumonia e somente 120 COVID | A loss global pode ser dominada pelas classes majoritárias, reduzindo sensibilidade à classe COVID |
| **Split 80/20 sem estratificação** | A proporção das classes não é garantida entre treino e validação | Métricas podem variar excessivamente e a minoria pode ficar mal representada |
| **ResNet-18 treinada do zero** | 1.200 imagens é um conjunto pequeno para aprender toda a representação profunda do zero | Maior risco de overfitting e generalização fraca |
| **Sem augmentation** | O modelo vê pouca variação de posicionamento, escala, contraste e aquisição | Menor robustez a variações plausíveis das radiografias |
| **Learning rate fixo de 0.01** | Não existe adaptação do passo durante o treinamento | Pode haver convergência instável ou atualização excessiva dos pesos |
| **Avaliação apenas por accuracy** | A classe Normal domina o dataset | Um modelo pode parecer forte globalmente mesmo deixando passar muitos COVID |
| **93% treino vs 61% validação** | Grande gap entre treino e validação | Forte sinal de overfitting / baixa capacidade de generalização |

### Evidência produzida neste notebook

Nosso baseline reforça especificamente o problema da métrica:

```text
Accuracy global = 95,00%
COVID Recall    = 55,56%
```

Portanto uma accuracy alta não garante sensibilidade adequada à classe minoritária.

# Parte H — Plano integrado de melhoria

## 22. Modelo

Uma melhoria sobre o projeto original é usar **transfer learning**.

Em vez de aprender todos os filtros do zero:

```text
pesos aleatórios
→ 1.200 imagens
```

usamos:

```text
ResNet-18 pré-treinada
→ conhecimento visual inicial
→ fine-tuning no raio-X
```

Uma evolução futura seria avaliar pesos pré-treinados em domínio médico/radiológico, porque ImageNet ainda possui domain shift em relação a radiografias.

## 23. Métricas

Accuracy deve ser apenas uma das métricas.

A avaliação deveria acompanhar pelo menos:

```text
Recall / sensibilidade por classe
Precision por classe
F1 por classe
Macro F1
Macro Recall
Matriz de confusão
```

Para COVID, o recall merece destaque porque mede:

\[
Recall =
rac{TP}{TP + FN}
\]

ou seja, entre todos os casos COVID reais, quantos foram encontrados.

Em triagem, os **falsos negativos** devem ser explicitamente analisados.

## 24. Dados e split

Recomendações:

```text
split estratificado
+
teste completamente separado
+
deduplicação antes do split
+
validação externa
```

Se existirem informações de paciente/hospital, o ideal é que o split seja feito também respeitando essas unidades, evitando que imagens altamente correlacionadas do mesmo paciente ou instituição vazem entre treino e teste.

## 25. Augmentation real

Transformações médicas devem ser plausíveis.

Neste projeto usamos pequenas:

- rotações;
- translações;
- alterações de escala;
- brilho;
- contraste.

O objetivo não é produzir imagens visualmente variadas a qualquer custo, mas introduzir variações compatíveis com aquisição e posicionamento sem distorcer estruturas anatômicas.

## 26. Augmentation sintética

O experimento mostra que a geração sintética deve possuir critérios de qualidade antes de ser usada como treino.

Antes de adoção, uma futura abordagem deveria avaliar:

```text
diversidade
qualidade visual
similaridade com distribuição real
presença de artefatos
efeito downstream no classificador
```

Possíveis linhas futuras:

- WGAN-GP;
- arquiteturas generativas mais fortes;
- modelos de difusão;
- geração em maior resolução;
- feature-based evaluation entre real e sintético;
- avaliação por especialistas do domínio.

O ponto principal é que **melhoria da loss adversarial não é suficiente**. O dado sintético só é útil se melhorar a tarefa downstream ou representar melhor a distribuição real.

## 27. Critério de adoção clínica

Nenhum modelo deste notebook deve ser considerado clinicamente adequado apenas pela accuracy.

Um processo de adoção exigiria, entre outros pontos:

```text
validação externa
↓
dados de outras instituições/equipamentos

análise de falsos negativos
↓
especialmente para a classe de interesse

avaliação por especialistas
↓
revisão clínica das decisões e erros

teste prospectivo
↓
comportamento no fluxo real

monitoramento
↓
drift e desempenho pós-implantação
```

Nosso próprio resultado ilustra o problema:

```text
95% accuracy
mas
55,56% recall COVID
```

Por isso, o critério de adoção precisa ser definido em termos de segurança, sensibilidade e validação clínica, e não apenas por uma métrica global alta.

# Parte I — Conceitos para estudo

## 28. Resumo do fluxo completo

```text
Dataset de raio-X
        ↓
deduplicação + amostragem
        ↓
split estratificado
        ↓
ResNet-18 baseline
        ↓
COVID Recall = 55,56%

---------------------------------

Treino real apenas
        ↓
Conditional GAN
        ↓
mode collapse detectado
        ↓
mitigação 1
LR D menor + label smoothing + instance noise
        ↓
equilíbrio G/D melhora
mas diversidade continua baixa
        ↓
Mode-Seeking Regularization
        ↓
melhoria parcial da diversidade
        ↓
84 COVID sintéticas
        ↓
ResNet-18 novamente
        ↓
COVID Recall = 38,89%
```

## 29. Vocabulário essencial

### GAN
Rede generativa adversarial formada por Generator e Discriminator.

### Conditional GAN
GAN que recebe informação de classe durante geração e discriminação.

### Latent vector `z`
Vetor aleatório que representa a fonte de variação usada pelo Generator.

### Embedding de classe
Representação vetorial aprendível do label usada para condicionar a geração.

### ConvTranspose2d
Operação usada pelo Generator para aumentar progressivamente a resolução espacial.

### Tanh
Ativação final do Generator, produzindo valores no intervalo aproximado `[-1, 1]`.

### BCEWithLogitsLoss
Binary Cross-Entropy aplicada de forma numericamente estável diretamente sobre logits.

### Mode collapse
Situação em que diferentes entradas latentes geram poucas saídas muito semelhantes.

### Label smoothing
Substitui um target real rígido como `1.0` por valor menor, como `0.9`, reduzindo confiança excessiva do Discriminator.

### Instance noise
Ruído adicionado temporariamente às entradas do Discriminator para suavizar a separação real/fake no início.

### Mode-Seeking Regularization
Termo adicional que incentiva diferentes vetores latentes a produzir diferentes imagens.

### Synthetic augmentation
Uso de imagens geradas artificialmente como exemplos adicionais de treinamento.

### Recall
Proporção dos casos positivos reais que foram encontrados pelo modelo.

### Falso negativo
Caso realmente positivo que o modelo classificou como outra classe.

## 30. Conclusões

O estudo de caso demonstrou que:

1. **accuracy global é insuficiente** para avaliar uma classe minoritária crítica;
2. transfer learning melhora a metodologia em relação a treinar uma ResNet pequena do zero em apenas 1.200 imagens;
3. GANs possuem treinamento instável e exigem diagnóstico visual e quantitativo;
4. reduzir dominância do Discriminator não garante resolver mode collapse;
5. Mode-Seeking aumentou parcialmente a diversidade final;
6. diversidade isolada não mede realismo ou utilidade;
7. adicionar 84 imagens sintéticas reduziu o COVID recall de **55,56% para 38,89%**;
8. dados sintéticos precisam ser avaliados pelo impacto downstream, não apenas pela aparência;
9. qualquer adoção clínica exigiria validação muito mais rigorosa do que este experimento acadêmico.

## 31. Checklist da rubrica — Atividade 4.1

- [x] Diagnóstico de pelo menos 5 problemas técnicos — foram identificados 7
- [x] Impacto esperado de cada problema discutido
- [x] Conditional GAN para domínio médico
- [x] Generator condicional
- [x] Discriminator condicional
- [x] Training loop adversarial correto
- [x] Validação/teste excluídos do treinamento da GAN
- [x] Mode collapse diagnosticado quantitativamente
- [x] Mode collapse diagnosticado visualmente
- [x] Primeira estratégia de mitigação aplicada
- [x] Learning rate assimétrico
- [x] Label smoothing
- [x] Instance noise
- [x] Segunda estratégia: Mode-Seeking Regularization
- [x] Evidência quantitativa da melhoria parcial de diversidade
- [x] Limitação da mitigação discutida
- [x] 84 imagens COVID sintéticas geradas
- [x] Mesmo classificador treinado com e sem sintéticos
- [x] Mesma validação e mesmo teste real
- [x] COVID recall comparado experimentalmente
- [x] Resultado negativo da augmentation reportado honestamente
- [x] Plano integrado cobrindo modelo
- [x] Plano integrado cobrindo métricas
- [x] Plano integrado cobrindo augmentation sintética
- [x] Critério de adoção clínica discutido

> O estudo de caso de **tráfego urbano (Atividade 4.2)** é entregue somente no relatório técnico, conforme o enunciado, e não faz parte deste notebook.